# Day 2: Local LLM, GPU Usage, and Hugging Face Workflow

day 1 was all about how you phrase a prompt. today's about actually running an LLM yourself instead of relying on a hosted playground somewhere. installed Ollama with Homebrew, pulled a small model (llama3.2:3b, about 2GB), and ran real inference against it locally, no API key, nothing leaving this machine after the model finished downloading.

second half compares that against loading a model directly through Hugging Face, since it's a genuinely different way of doing the same basic thing, and closes out with chat templates, which turned into a bigger deal than expected once one of the runs actually broke because of it.

In [1]:
import requests
import json

## Ollama runs as a local background service on port 11434 by default
## no API key, no internet call after the model is pulled -- fully local inference

OLLAMA_URL = "http://localhost:11434/api/generate"

def ask_ollama(prompt, model="llama3.2:3b"):
    response = requests.post(OLLAMA_URL, json={
        "model": model,
        "prompt": prompt,
        "stream": False,
    })
    return response.json()["response"]

print("Asking local llama3.2:3b via Ollama...")
answer = ask_ollama("What is the difference between a CPU and a GPU, in two sentences?")
print(answer)
##successful run with appropriate output observed
##side note: the memory is limited to 2023, tested with an example asking about current Tesla lineup and it gave older info but declared it is updated in knowledge till 2023

Asking local llama3.2:3b via Ollama...


A CPU (Central Processing Unit) is a processor that executes most instructions in a computer, handling calculations, data processing, and logical operations, making it the brain of the computer. A GPU (Graphics Processing Unit) is a specialized processor designed to handle graphics, video, and compute tasks, providing high-speed performance for tasks like gaming, video editing, and scientific simulations.


that Tesla test is worth calling out on its own. every LLM has a training cutoff baked into it, a point where its knowledge just stops, and it doesn't reliably warn you before answering with something outdated. it'll answer confidently either way. asked it about Tesla's current lineup and got an answer that was accurate as of a while ago, not now, which is the same root problem as Day 1's hallucinated Dow Jones number just showing up in a different shape, the model has no live connection to the world, only whatever got frozen into it during training. this is basically the entire reason RAG exists later this week, instead of trying to make a model "more current" through training, you hand it real current documents at question time and let it answer from those instead of memory.

In [2]:
##-----Hugging face: loading a model directly VS Ollama running it -----
from transformers import pipeline

print("\nLoading TinyLlama directly via Hugging Face...")
hf_pipe = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-chat-v1.0")

def ask_hf_raw(prompt, max_new_tokens=60):
    output = hf_pipe(prompt, max_new_tokens=max_new_tokens)
    return output[0]["generated_text"]

def ask_hf_chat(user_message, max_new_tokens=60):
    messages = [
        {"role": "user", "content": user_message},
    ]
    output = hf_pipe(messages, max_new_tokens=max_new_tokens)
    return output[0]["generated_text"][-1]["content"]

print("\n--- Raw prompt, no chat template ---")
print(ask_hf_raw("What is the difference between CPU and GPU, in two sentences?"))

print("\n--- Same question, using the chat template ---")
print(ask_hf_chat("What is the difference between CPU and GPU, in two sentences?", max_new_tokens=120))

print("\n--- Raw prompt, no chat template (Overwatch) ---")
print(ask_hf_raw("Describe the video game Overwatch in two sentences."))

print("\n--- Same question, using the chat template (Overwatch) ---")
print(ask_hf_chat("Describe the video game Overwatch in two sentences.", max_new_tokens=120))

## raw prompt (no chat template) never actually answered either question, it just kept
## generating more similar-looking questions instead ("How does a computer work...",
## "2. How does the combat system work?") -- pure next-token continuation, same base-model
## behavior as Day 6's GPT-2, since TinyLlama-Chat was never told this was a turn to respond to
##
## chat template version actually answered both questions for real (correct CPU/GPU breakdown,
## correct Overwatch facts: Blizzard, first-person shooter, 2016) -- first attempt got cut off
## mid-sentence at max_new_tokens=60, bumped to 120 to let both answers finish completely
##
## real proof that chat templates aren't just formatting preference, they're the difference
## between a chat-tuned model recognizing "respond to this" vs. just continuing raw text

## note: outputs varied between runs since pipeline() samples by default (no fixed seed/greedy) --
## same randomness concept as Day 1's temperature section, just showing up unannounced here
##
## real hallucination caught in the raw Overwatch answer: it described "orcs, trolls" as being
## in Overwatch, which is actually Warcraft, a different Blizzard game entirely -- model blended
## two franchises together confidently. chat template fixed the *structure* problem (answering
## at all) but not this -- a model can follow format perfectly and still get facts wrong

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading TinyLlama directly via Hugging Face...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6925.56it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


...

What is the difference between CPU and GPU, in two sentences?

--- Same question, using the chat template ---


[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1. CPU (Central Processing Unit): The primary component of a computer that executes instructions and processes data, while GPU (Graphics Processing Unit) is responsible for rendering visuals or graphics.

2. In two sentences: CPUs are the brains of a computer, while GPUs are the graphics card.

--- Raw prompt, no chat template (Overwatch) ---
Describe the video game Overwatch in two sentences.

--- Same question, using the chat template (Overwatch) ---


Overwatch is a competitive multiplayer online battle arena (MOBA) game where players control teams of four characters, each with unique abilities and roles, vying for control of a fictional overwatch base that must be defended against an opposing team. The game features a fast-paced, action-packed combat system and a diverse character roster with a focus on teamwork and strategic planning.


this section started as a bug and turned into the actual lesson. first attempt at the raw prompt call had a typo, `max_new_tokens=60` ended up typed inside the string instead of as its own argument, so the model wasn't asked a clean question at all, it was handed a broken chunk of text that looked like leftover Python code and it responded by generating more code-looking nonsense. fixed the typo, but the raw version kept misbehaving anyway even with clean syntax, since `pipeline("text-generation")` sends a prompt completely raw with no structure, and TinyLlama-Chat was trained to expect its input wrapped in a specific `<|user|>` / `<|assistant|>` format. without that wrapping the model doesn't reliably know it's supposed to answer at all, sometimes it just continues the text in some plausible-looking direction instead, which is exactly what happened on both the CPU/GPU and Overwatch prompts.

switching to `ask_hf_chat`, which passes a list of role/content dicts instead of a bare string, lets the pipeline apply that chat template automatically, and both answers actually became real answers instead of continuations. same model, same question, only difference is whether the input got formatted the way the model was fine-tuned to expect.

CPU handles general purpose stuff, one instruction at a time, but does complex individual operations fast. GPU has thousands of smaller cores built to do the same operation on a ton of numbers all at once, which is basically what a neural network is, endless matrix multiplication. that's why LLMs run at real speed on a GPU and not a CPU.

VRAM is the GPU's own dedicated memory, separate from regular system RAM. a model's weights have to fit inside VRAM to run on the GPU, that's the actual bottleneck, not raw processing speed. a 7B parameter model at full precision needs roughly 28GB of VRAM just to load, more than most consumer GPUs even have. CUDA is Nvidia's software layer that lets code actually talk to their GPUs and run that parallel math, it's the industry standard most ML libraries are built assuming exists.

Quantization is the actual fix for that VRAM problem. a model's weights normally get stored as 32-bit or 16-bit floating point numbers, quantization shrinks each number down to fewer bits, 8-bit or 4-bit, which directly shrinks how much memory the whole model takes up. a model that needs 28GB at full precision might only need 7GB at 4-bit. the trade is less memory and faster loading in exchange for some accuracy lost from rounding every number to a coarser scale, and it's the whole reason running big models on a regular laptop is possible at all.

Llama.cpp is the actual engine under a lot of this, a C++ project built to run LLMs efficiently on regular hardware, no GPU required. it's not something you use directly, it's what other tools get built on top of. Ollama is a command-line tool built on that same kind of engine, wraps up model downloading, serving, and running behind one clean local API, so you don't have to think about the setup underneath. LM Studio is the same basic idea as Ollama but with a full GUI on top, browsing and downloading models and chatting with them directly in an app instead of code. vLLM is a different category entirely, built for production serving, handling many users hitting a model at once with high throughput, more what a company runs behind a real API endpoint than what one person runs locally.

## takeaway

closes out with Ollama and Hugging Face turning out to be two genuinely different levels of control over the same basic idea, running a model yourself. Ollama abstracts away basically everything, one API call and you get an answer, model management handled for you. Hugging Face's pipeline hands you the actual model, actual tokenizer, actual generation settings, and expects you to know what each one needs, chat template included, which today's whole middle section ended up proving the hard way.

also worth keeping the Tesla and Overwatch findings side by side, one is a knowledge cutoff problem, the model genuinely doesn't know what happened after training ended. the other is a straight up hallucination, the model blending two different games together with total confidence. different failure, same underlying issue: nothing grounding the model in anything real. RAG later this week is built to fix exactly that.